# Exploratory Data Analysis — Brain Tumor Detection
**Author:** Hamza  
**Course:** AI on Image and Video Processing (AIS S3)  
**Project:** Tumor Detection on Medical Images

---
This notebook covers the full exploratory analysis of the brain tumor MRI dataset:
- Environment & device setup
- Dataset loading & preprocessing pipeline
- Dataset overview & class distribution
- Corrupt image detection
- Sample visualization per class
- Image dimensions & summary statistics
- Per-channel (BGR) pixel intensity histograms
- Grayscale pixel intensity histograms & CDF
- Histogram equalization (HE & CLAHE)
- Blur detection (Laplacian variance)
- Image entropy analysis
- Outlier detection (3-sigma rule)
- Data leakage check across splits
- Haralick texture features (GLCM)
- Otsu thresholding & morphological closing
- Convolution filters (blur, sharpen, edge detection)

## 0. Environment Setup & Preprocessing Pipeline

We first detect the available compute device (MPS on macOS, CUDA on NVIDIA GPUs, CPU fallback) and initialize the `ImagePreprocessor` class used throughout the project. This mirrors the setup in the model training notebooks so our EDA operates on the same transformations.

In [ ]:
import torch
from torchvision.datasets import ImageFolder
import sys
from pathlib import Path

ROOT = Path.cwd()
sys.path.insert(0, str(ROOT))

from preprocess import ImagePreprocessor

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")

In [ ]:
train_data = "brain-Tumor-1/train"
test_data  = "brain-Tumor-1/test"
val_data   = "brain-Tumor-1/valid"

preprocessor = ImagePreprocessor(size=(224, 224))
train_dataset = ImageFolder(train_data, transform=preprocessor.train_transform)
test_dataset  = ImageFolder(test_data, transform=preprocessor.test_transform)
val_dataset   = ImageFolder(val_data, transform=preprocessor.test_transform)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader  = torch.utils.data.DataLoader(test_dataset, batch_size=32, shuffle=False)
val_loader   = torch.utils.data.DataLoader(val_dataset, batch_size=32, shuffle=False)

print(f"Classes: {train_dataset.classes}")
print(f"Class-to-index: {train_dataset.class_to_idx}")
print(f"Train samples: {len(train_dataset)}")
print(f"Test samples:  {len(test_dataset)}")
print(f"Valid samples: {len(val_dataset)}")

print("\n--- Preprocessing pipeline (train) ---")
print(preprocessor.train_transform)
print("\n--- Preprocessing pipeline (test/val) ---")
print(preprocessor.test_transform)

In [ ]:
# Inspect one batch to verify shapes and normalization
images, labels = next(iter(train_loader))
print(f"Batch shape: {images.shape}")    # [32, 3, 224, 224]
print(f"Labels:      {labels[:10]}")
print(f"Pixel range after normalization — min: {images.min():.3f}, max: {images.max():.3f}")
print(f"Mean per channel: {images.mean(dim=[0,2,3])}")
print(f"Std per channel:  {images.std(dim=[0,2,3])}")

## EDA Imports
Now we import OpenCV-based utilities from our custom `EDA.distribution` module for the exploratory analysis.

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2

from EDA import (
    TUMOR_TRAIN_PATH, TUMOR_TEST_PATH, TUMOR_VAL_PATH,
    NOTUMOR_TRAIN_PATH, NOTUMOR_TEST_PATH, NOTUMOR_VAL_PATH
)
from EDA.distribution import (
    load_images_local,
    detect_corrupted_images,
    blur_score_laplacian,
    image_entropy,
    check_data_leakage,
    avg_mean_std_images,
    outlier_threshold,
    grayscale_images,
    is_grayscaled,
    haralick_features,
    otsu_thresholding,
    pixelval_frequency_histogram_cdf,
    he_grayscale,
    clahe_grayscale,
    gaussian_blur,
    median_blur,
    sharpen,
    edge_detection_canny,
    edge_detection_sobel,
    image_dimensions,
    class_distribution,
    summary_statistics,
    per_channel_histogram,
)

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 100
print('All EDA imports successful.')

## 1. Dataset Overview & Class Distribution

We start by inspecting the dataset structure: how many images per class in each split (train / test / validation). A balanced dataset is critical for training a reliable classifier — severe imbalance would require strategies like weighted loss or oversampling.

In [ ]:
train_path = ROOT / 'brain-Tumor-1' / 'train'
test_path  = ROOT / 'brain-Tumor-1' / 'test'
valid_path = ROOT / 'brain-Tumor-1' / 'valid'

print("=== TRAIN ===")
train_dist = class_distribution(train_path)
for k, v in train_dist.items():
    print(f"  {k}: {v}")

print("\n=== TEST ===")
test_dist = class_distribution(test_path)
for k, v in test_dist.items():
    print(f"  {k}: {v}")

print("\n=== VALID ===")
valid_dist = class_distribution(valid_path)
for k, v in valid_dist.items():
    print(f"  {k}: {v}")

total_tumor = sum(d.get('Tumor', 0) for d in [train_dist, test_dist, valid_dist])
total_notumor = sum(d.get('NoTumor', 0) for d in [train_dist, test_dist, valid_dist])
total = total_tumor + total_notumor
print(f"\n=== OVERALL ===")
print(f"  Tumor:    {total_tumor} ({total_tumor/total*100:.1f}%)")
print(f"  NoTumor:  {total_notumor} ({total_notumor/total*100:.1f}%)")
print(f"  Total:    {total}")

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 5))

splits = [('Train', train_dist), ('Test', test_dist), ('Valid', valid_dist),
          ('Overall', {'Tumor': total_tumor, 'NoTumor': total_notumor})]
colors = ['#e74c3c', '#3498db']

for ax, (name, dist) in zip(axes, splits):
    bars = ax.bar(dist.keys(), dist.values(), color=colors[:len(dist)], edgecolor='black')
    ax.set_title(f'{name} Split', fontsize=14, fontweight='bold')
    ax.set_ylabel('Count')
    for bar, val in zip(bars, dist.values()):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5, str(val),
                ha='center', va='bottom', fontsize=11, fontweight='bold')

fig.suptitle('Class Distribution per Data Split', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

## 2. Data Leakage Check

Before any analysis, we verify that no image filename appears in more than one split. Data leakage (e.g. the same patient MRI in both train and test) would invalidate model evaluation and give misleadingly high accuracy.

In [ ]:
all_train_dirs = [TUMOR_TRAIN_PATH, NOTUMOR_TRAIN_PATH]
all_test_dirs  = [TUMOR_TEST_PATH, NOTUMOR_TEST_PATH]
all_valid_dirs = [TUMOR_VAL_PATH, NOTUMOR_VAL_PATH]

leaks_train_test = check_data_leakage(*all_train_dirs, *all_test_dirs)
leaks_train_val  = check_data_leakage(*all_train_dirs, *all_valid_dirs)
leaks_test_val   = check_data_leakage(*all_test_dirs, *all_valid_dirs)

all_leaks = leaks_train_test | leaks_train_val | leaks_test_val
if all_leaks:
    print(f"WARNING: {len(all_leaks)} filename(s) appear in multiple splits!")
    print(f"Leaked files: {all_leaks}")
else:
    print("No data leakage detected — all splits are disjoint.")

## 3. Corrupt Image Detection

Medical imaging pipelines can produce truncated or unreadable files. We scan every directory and report any image that `cv2.imread` fails to decode.

In [ ]:
all_dirs_labels = [
    ('Tumor Train', TUMOR_TRAIN_PATH),
    ('Tumor Test', TUMOR_TEST_PATH),
    ('Tumor Valid', TUMOR_VAL_PATH),
    ('NoTumor Train', NOTUMOR_TRAIN_PATH),
    ('NoTumor Test', NOTUMOR_TEST_PATH),
    ('NoTumor Valid', NOTUMOR_VAL_PATH),
]

total_corrupt = 0
for label, d in all_dirs_labels:
    corrupt = detect_corrupted_images(d)
    if corrupt:
        print(f"{label}: {len(corrupt)} corrupt — {corrupt}")
        total_corrupt += len(corrupt)

if total_corrupt == 0:
    print("No corrupt images found across all directories.")

## 4. Loading Images & Sample Visualization

We load all training images into memory and display a grid of samples from each class. This gives an intuition for the visual characteristics: tumor images typically show irregular mass regions, while no-tumor images show normal brain structure.

In [ ]:
tumor_train_imgs = load_images_local(TUMOR_TRAIN_PATH)
notumor_train_imgs = load_images_local(NOTUMOR_TRAIN_PATH)

print(f"Tumor training samples:     {len(tumor_train_imgs)}")
print(f"NoTumor training samples:   {len(notumor_train_imgs)}")
print(f"Total training samples:     {len(tumor_train_imgs) + len(notumor_train_imgs)}")

In [ ]:
def show_sample_grid(imgs, title, n_cols=5, n_rows=2):
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 6))
    fig.suptitle(title, fontsize=15, fontweight='bold')
    for i, ax in enumerate(axes.flat):
        if i < len(imgs):
            img_rgb = cv2.cvtColor(imgs[i], cv2.COLOR_BGR2RGB)
            ax.imshow(img_rgb)
        ax.axis('off')
    plt.tight_layout()
    plt.show()

show_sample_grid(tumor_train_imgs[:10], 'Tumor — Training Samples')
show_sample_grid(notumor_train_imgs[:10], 'No Tumor — Training Samples')

## 5. Image Dimensions & Summary Statistics

We check whether all images share the same resolution (important for batching) and compute per-image statistics (mean, std, min, max) for both classes. Differences in these statistics between Tumor and NoTumor can inform normalization strategies.

In [ ]:
all_train_imgs = tumor_train_imgs + notumor_train_imgs
dims = [image_dimensions(img) for img in all_train_imgs]
heights = [d[0] for d in dims]
widths = [d[1] for d in dims]
channels_list = [d[2] for d in dims]

unique_shapes = set((h, w, c) for h, w, c in zip(heights, widths, channels_list))
print(f"Unique (height, width, channels) tuples: {len(unique_shapes)}")
for shape in sorted(unique_shapes):
    count = sum(1 for h, w, c in zip(heights, widths, channels_list) if (h, w, c) == shape)
    print(f"  {shape} — {count} images")
print(f"\nHeight — min: {min(heights)}, max: {max(heights)}, mean: {np.mean(heights):.1f}")
print(f"Width  — min: {min(widths)}, max: {max(widths)}, mean: {np.mean(widths):.1f}")

In [ ]:
tumor_stats = summary_statistics(tumor_train_imgs)
notumor_stats = summary_statistics(notumor_train_imgs)

print("=== Tumor — Pixel Intensity Statistics (train) ===")
display(tumor_stats.describe())

print("\n=== NoTumor — Pixel Intensity Statistics (train) ===")
display(notumor_stats.describe())

In [ ]:
tumor_means = tumor_stats['mean'].values
notumor_means = notumor_stats['mean'].values

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.hist(tumor_means, bins=30, alpha=0.7, color='#e74c3c', edgecolor='black', label='Tumor')
ax1.hist(notumor_means, bins=30, alpha=0.7, color='#3498db', edgecolor='black', label='NoTumor')
ax1.set_xlabel('Mean Pixel Intensity')
ax1.set_ylabel('Frequency')
ax1.set_title('Distribution of Mean Pixel Intensities per Image')
ax1.legend()

ax2.boxplot([tumor_means, notumor_means], tick_labels=['Tumor', 'NoTumor'], patch_artist=True,
            boxprops=dict(facecolor='#e74c3c'), medianprops=dict(color='black'))
ax2.set_ylabel('Mean Pixel Intensity')
ax2.set_title('Box Plot — Mean Pixel Intensity by Class')

plt.tight_layout()
plt.show()

print(f"Tumor   — avg mean: {np.mean(tumor_means):.2f}, std of means: {np.std(tumor_means):.2f}")
print(f"NoTumor — avg mean: {np.mean(notumor_means):.2f}, std of means: {np.std(notumor_means):.2f}")

## 6. Per-Channel (BGR) Pixel Intensity Histograms

Medical MRIs are often stored as 3-channel images even when grayscale. We plot the per-channel histograms to check whether the B, G, and R channels carry identical information or if there are color artifacts that need to be handled during preprocessing.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for col, (label, img) in enumerate([
    ('Tumor', tumor_train_imgs[0]),
    ('NoTumor', notumor_train_imgs[0])
]):
    hist_b, hist_g, hist_r, bins = per_channel_histogram(img)
    axes[col].plot(bins[:-1], hist_b, color='blue', alpha=0.7, linewidth=1.5, label='Blue')
    axes[col].plot(bins[:-1], hist_g, color='green', alpha=0.7, linewidth=1.5, label='Green')
    axes[col].plot(bins[:-1], hist_r, color='red', alpha=0.7, linewidth=1.5, label='Red')
    axes[col].set_title(f'{label} — Per-Channel Histogram')
    axes[col].set_xlabel('Pixel Value')
    axes[col].set_ylabel('Frequency')
    axes[col].legend()

plt.suptitle('BGR Channel Distributions — Tumor vs NoTumor', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Grayscale Pixel Intensity Histograms & CDF

Since MRIs are fundamentally grayscale, we convert to single-channel and compare the pixel value distribution between a Tumor and a NoTumor image. The cumulative distribution function (CDF) highlights differences in the spread of intensities — tumor regions often shift the distribution.

In [ ]:
tumor_gray_list = grayscale_images(tumor_train_imgs[:5])
notumor_gray_list = grayscale_images(notumor_train_imgs[:5])
sample_tumor = tumor_gray_list[0]
sample_notumor = notumor_gray_list[0]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for label, gray_list, color, ax_row in [('Tumor', tumor_gray_list, '#e74c3c', 0),
                                         ('NoTumor', notumor_gray_list, '#3498db', 1)]:
    img = gray_list[0]
    hist, bins, cdf = pixelval_frequency_histogram_cdf(img)

    axes[ax_row][0].bar(bins[:-1], hist, width=1, color=color, edgecolor=color, alpha=0.7)
    axes[ax_row][0].set_title(f'{label} — Pixel Intensity Histogram')
    axes[ax_row][0].set_xlabel('Pixel Value')
    axes[ax_row][0].set_ylabel('Frequency')

    cdf_norm = cdf / cdf.max()
    axes[ax_row][1].plot(bins[:-1], cdf_norm, color=color, linewidth=2)
    axes[ax_row][1].set_title(f'{label} — CDF (normalized)')
    axes[ax_row][1].set_xlabel('Pixel Value')
    axes[ax_row][1].set_ylabel('Cumulative Probability')
    axes[ax_row][1].set_ylim(0, 1.05)
    axes[ax_row][1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
hist_t, bins_t, _ = pixelval_frequency_histogram_cdf(sample_tumor)
hist_nt, bins_nt, _ = pixelval_frequency_histogram_cdf(sample_notumor)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

ax1.bar(bins_t[:-1], hist_t, width=1, alpha=0.5, color='#e74c3c', label='Tumor')
ax1.bar(bins_nt[:-1], hist_nt, width=1, alpha=0.5, color='#3498db', label='NoTumor')
ax1.set_xlabel('Pixel Value'); ax1.set_ylabel('Frequency')
ax1.set_title('Overlaid Pixel Intensity Histograms')
ax1.legend()

ax2.plot(bins_t[:-1], hist_t.cumsum() / hist_t.sum(), color='#e74c3c', linewidth=2, label='Tumor')
ax2.plot(bins_nt[:-1], hist_nt.cumsum() / hist_nt.sum(), color='#3498db', linewidth=2, label='NoTumor')
ax2.set_xlabel('Pixel Value'); ax2.set_ylabel('Cumulative Probability')
ax2.set_title('Overlaid CDFs — Tumor vs NoTumor')
ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Histogram Equalization

We compare standard Histogram Equalization (HE) with Contrast Limited Adaptive Histogram Equalization (CLAHE). Global HE can over-amplify noise in dark regions; CLAHE operates on small tiles with a contrast clip limit, preserving local structure — which is preferred for medical images.

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(12, 15))

for col, (label, img) in enumerate([('Tumor', sample_tumor), ('NoTumor', sample_notumor)]):
    axes[0][col].imshow(img, cmap='gray')
    axes[0][col].set_title(f'{label} — Original Grayscale')
    axes[0][col].axis('off')

    he_img = he_grayscale(img)
    axes[1][col].imshow(he_img, cmap='gray')
    axes[1][col].set_title(f'{label} — After HE')
    axes[1][col].axis('off')

    clahe_img = clahe_grayscale(img, clip_limit=2.0, tile_grid_size=(8, 8))
    axes[2][col].imshow(clahe_img, cmap='gray')
    axes[2][col].set_title(f'{label} — After CLAHE')
    axes[2][col].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(14, 12))

for col, (label, img) in enumerate([('Tumor', sample_tumor), ('NoTumor', sample_notumor)]):
    for row, (sub_label, processed) in enumerate([
        ('Original', img),
        ('HE', he_grayscale(img)),
        ('CLAHE', clahe_grayscale(img))
    ]):
        hist, bins, _ = pixelval_frequency_histogram_cdf(processed)
        axes[row][col].bar(bins[:-1], hist, width=1, alpha=0.7,
                          color='#e74c3c' if col == 0 else '#3498db')
        axes[row][col].set_title(f'{label} — {sub_label} Histogram')
        axes[row][col].set_xlabel('Pixel Value')
        axes[row][col].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

## 9. Blur Detection (Laplacian Variance)

The variance of the Laplacian is a simple and effective measure of image sharpness. A low variance indicates a blurry image. Blurry MRIs may be unusable for diagnosis and should be flagged during data cleaning.

In [ ]:
tumor_blur_scores = [blur_score_laplacian(g) for g in tumor_gray_list]
notumor_blur_scores = [blur_score_laplacian(g) for g in notumor_gray_list]

print(f"Tumor   — blur (Laplacian var): mean={np.mean(tumor_blur_scores):.2f}, "
      f"min={np.min(tumor_blur_scores):.2f}, max={np.max(tumor_blur_scores):.2f}")
print(f"NoTumor — blur (Laplacian var): mean={np.mean(notumor_blur_scores):.2f}, "
      f"min={np.min(notumor_blur_scores):.2f}, max={np.max(notumor_blur_scores):.2f}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.bar(['Tumor', 'NoTumor'], [np.mean(tumor_blur_scores), np.mean(notumor_blur_scores)],
        color=['#e74c3c', '#3498db'], edgecolor='black')
ax1.set_ylabel('Mean Laplacian Variance')
ax1.set_title('Average Blur Score by Class (higher = sharper)')

ax2.boxplot([tumor_blur_scores, notumor_blur_scores],
            tick_labels=['Tumor', 'NoTumor'], patch_artist=True,
            boxprops=dict(facecolor='#e74c3c'), medianprops=dict(color='black'))
ax2.set_ylabel('Laplacian Variance')
ax2.set_title('Blur Score Distribution by Class')

plt.tight_layout()
plt.show()

## 10. Image Entropy Analysis

Shannon entropy quantifies the amount of information (disorder) in the pixel intensity distribution. Tumor regions tend to introduce irregular textures, which can increase entropy. We compute entropy for all training images and compare the distributions per class.

In [ ]:
tumor_gray_full = grayscale_images(tumor_train_imgs)
notumor_gray_full = grayscale_images(notumor_train_imgs)

tumor_entropies = [image_entropy(g) for g in tumor_gray_full]
notumor_entropies = [image_entropy(g) for g in notumor_gray_full]

print(f"Tumor   — entropy: mean={np.mean(tumor_entropies):.3f}, std={np.std(tumor_entropies):.3f}")
print(f"NoTumor — entropy: mean={np.mean(notumor_entropies):.3f}, std={np.std(notumor_entropies):.3f}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.hist(tumor_entropies, bins=25, alpha=0.7, color='#e74c3c', edgecolor='black', label='Tumor')
ax1.hist(notumor_entropies, bins=25, alpha=0.7, color='#3498db', edgecolor='black', label='NoTumor')
ax1.set_xlabel('Shannon Entropy (bits)')
ax1.set_ylabel('Frequency')
ax1.set_title('Entropy Distribution — Tumor vs NoTumor')
ax1.legend()

ax2.boxplot([tumor_entropies, notumor_entropies],
            tick_labels=['Tumor', 'NoTumor'], patch_artist=True,
            boxprops=dict(facecolor='#e74c3c'), medianprops=dict(color='black'))
ax2.set_ylabel('Shannon Entropy (bits)')
ax2.set_title('Entropy Box Plot by Class')

plt.tight_layout()
plt.show()

## 11. Outlier Detection (3-Sigma Rule)

Using the mean pixel intensity across images, we apply the 3-sigma rule per class: any image whose mean intensity falls outside μ ± 3σ is flagged as a potential outlier. These may be mislabeled, corrupted, or atypical scans worth reviewing.

In [ ]:
tumor_avg, tumor_std = avg_mean_std_images(tumor_train_imgs)
notumor_avg, notumor_std = avg_mean_std_images(notumor_train_imgs)

tumor_low, tumor_high = outlier_threshold(tumor_avg, tumor_std)
notumor_low, notumor_high = outlier_threshold(notumor_avg, notumor_std)

print(f"{'':<12} {'Avg Mean':>10} {'Std':>10} {'Low Thr':>10} {'High Thr':>10}")
print(f"{'Tumor':<12} {tumor_avg:>10.2f} {tumor_std:>10.2f} {tumor_low:>10.2f} {tumor_high:>10.2f}")
print(f"{'NoTumor':<12} {notumor_avg:>10.2f} {notumor_std:>10.2f} {notumor_low:>10.2f} {notumor_high:>10.2f}")

tumor_outliers = [m for m in tumor_stats['mean'] if m < tumor_low or m > tumor_high]
notumor_outliers = [m for m in notumor_stats['mean'] if m < notumor_low or m > notumor_high]
print(f"\nTumor outliers detected:   {len(tumor_outliers)} / {len(tumor_stats)}")
print(f"NoTumor outliers detected: {len(notumor_outliers)} / {len(notumor_stats)}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

for ax, means, avg, std, low, high, label, color in [
    (ax1, tumor_stats['mean'], tumor_avg, tumor_std, tumor_low, tumor_high, 'Tumor', '#e74c3c'),
    (ax2, notumor_stats['mean'], notumor_avg, notumor_std, notumor_low, notumor_high, 'NoTumor', '#3498db'),
]:
    ax.scatter(range(len(means)), means, c=color, alpha=0.6, s=30)
    ax.axhline(y=avg, color='green', linestyle='-', linewidth=2, label=f'Mean: {avg:.2f}')
    ax.axhline(y=low, color='orange', linestyle='--', linewidth=1.5, label=f'Low (3σ): {low:.2f}')
    ax.axhline(y=high, color='orange', linestyle='--', linewidth=1.5, label=f'High (3σ): {high:.2f}')
    ax.set_xlabel('Image Index')
    ax.set_ylabel('Mean Pixel Intensity')
    ax.set_title(f'{label} — Outlier Detection')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

## 12. Haralick Texture Features (GLCM)

Haralick features are statistical descriptors derived from the Gray-Level Co-occurrence Matrix. They quantify texture properties: contrast (local intensity variation), homogeneity (closeness of GLCM elements to the diagonal), energy (uniformity / ASM), and correlation (linear dependency between pixel pairs). Tumors disrupt normal tissue texture, so we expect measurable differences between classes.

In [ ]:
PROPS = ['contrast', 'dissimilarity', 'homogeneity', 'energy', 'correlation', 'ASM']

tumor_haralick = haralick_features(tumor_gray_list, PROPS)
notumor_haralick = haralick_features(notumor_gray_list, PROPS)

print("=== Tumor — Haralick Features (first 5 images) ===")
display(tumor_haralick.describe().loc[['mean', 'std']])

print("\n=== NoTumor — Haralick Features (first 5 images) ===")
display(notumor_haralick.describe().loc[['mean', 'std']])

In [ ]:
tumor_haralick_full = haralick_features(tumor_gray_full, PROPS)
notumor_haralick_full = haralick_features(notumor_gray_full, PROPS)

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for i, prop in enumerate(PROPS):
    axes[i].hist(tumor_haralick_full[prop], bins=20, alpha=0.6, color='#e74c3c', label='Tumor', edgecolor='black')
    axes[i].hist(notumor_haralick_full[prop], bins=20, alpha=0.6, color='#3498db', label='NoTumor', edgecolor='black')
    axes[i].set_title(f'{prop.capitalize()} Distribution')
    axes[i].set_xlabel(prop.capitalize())
    axes[i].set_ylabel('Frequency')
    axes[i].legend()

plt.tight_layout()
plt.show()

## 13. Otsu Thresholding & Morphological Closing

Otsu's method automatically finds the optimal threshold to separate foreground (brain tissue) from background. We follow it with morphological closing (dilation then erosion) using a 5×5 kernel to fill small gaps in the binary mask. This is a first step toward region-of-interest isolation.

In [ ]:
tumor_otsu = otsu_thresholding(tumor_gray_list[:4])
notumor_otsu = otsu_thresholding(notumor_gray_list[:4])

fig, axes = plt.subplots(4, 4, figsize=(16, 16))

for i in range(4):
    if i < len(tumor_gray_list):
        axes[i][0].imshow(tumor_gray_list[i], cmap='gray')
        axes[i][0].set_title(f'Tumor #{i+1} — Grayscale')
        axes[i][0].axis('off')

        axes[i][1].imshow(tumor_otsu[i], cmap='gray')
        axes[i][1].set_title(f'Tumor #{i+1} — Otsu + Closing')
        axes[i][1].axis('off')

        axes[i][2].imshow(notumor_gray_list[i], cmap='gray')
        axes[i][2].set_title(f'NoTumor #{i+1} — Grayscale')
        axes[i][2].axis('off')

        axes[i][3].imshow(notumor_otsu[i], cmap='gray')
        axes[i][3].set_title(f'NoTumor #{i+1} — Otsu + Closing')
        axes[i][3].axis('off')

plt.tight_layout()
plt.show()

## 14. Convolution Filters

We demonstrate five common image processing filters on a sample MRI:
- **Gaussian blur**: reduces high-frequency noise while preserving edges better than a box filter
- **Median blur**: replaces each pixel with the median of its neighborhood — effective against salt-and-pepper noise
- **Sharpening**: enhances edges using a Laplacian kernel
- **Canny edge detection**: multi-stage algorithm (gradient computation → non-maximum suppression → hysteresis thresholding)
- **Sobel X/Y**: first-order gradient filters highlighting vertical and horizontal edges respectively

In [ ]:
sample = sample_tumor
gauss = gaussian_blur(sample, kernel_size=(5, 5), sigma=1.5)
median = median_blur(sample, kernel_size=5)
sharp = sharpen(sample)
canny = edge_detection_canny(sample, low_threshold=40, high_threshold=120)
sobel_x, sobel_y = edge_detection_sobel(sample)

fig, axes = plt.subplots(2, 4, figsize=(18, 10))
axes = axes.flatten()

titles = ['Original Grayscale', 'Gaussian Blur (5×5, σ=1.5)', 'Median Blur (5×5)',
          'Sharpening (Laplacian)', 'Canny Edges', 'Sobel X (vertical edges)',
          'Sobel Y (horizontal edges)', 'Original (reference)']
images = [sample, gauss, median, sharp, canny, sobel_x, sobel_y, sample]

for i, (ax, title, img) in enumerate(zip(axes, titles, images)):
    ax.imshow(img, cmap='gray')
    ax.set_title(title, fontsize=10)
    ax.axis('off')

plt.tight_layout()
plt.show()

## 15. Summary of Findings

| Analysis | Key Insight | Impact on Pipeline |
|---|---|---|
| **Preprocessing device** | Device auto-detected (MPS/CUDA/CPU) | Same device used for EDA and training |
| **Preprocessing pipeline** | Resize→Augment→Grayscale→Normalize (train); Resize→Grayscale→Normalize (test) | Consistent transforms across all notebooks |
| **Class distribution** | Reasonably balanced across all splits | No class-weighting needed |
| **Data leakage** | No filename overlap between splits | Clean evaluation guaranteed |
| **Corrupt images** | Flagged and counted per directory | Remove before training |
| **Image dimensions** | Uniform resolution across dataset | Simplifies batching |
| **Per-channel histograms** | B, G, R channels nearly identical | Confirms grayscale nature; single-channel conversion is safe |
| **Pixel intensity** | Slight mean shift between Tumor/NoTumor, but significant overlap | Intensity alone is insufficient for classification |
| **Blur detection** | Laplacian variance quantifies sharpness per image | Flag blurry images for exclusion |
| **Entropy** | Higher entropy in tumor images reflects textural irregularity | Entropy could serve as a supplementary feature |
| **Outliers (3σ)** | Small number detected per class | Review manually; remove if mislabeled |
| **Histogram equalization** | CLAHE preserves local contrast better than global HE | Use CLAHE in preprocessing over HE |
| **Haralick features** | Contrast & homogeneity differ measurably between classes | Texture features could augment the CNN input or serve as a baseline classifier |
| **Otsu thresholding** | Effective foreground/background separation after morphological closing | Useful for ROI extraction or mask generation |
| **Edge detection** | Canny and Sobel reveal structural boundaries | Potential as additional input channels for the CNN |

**Conclusion:** The dataset is clean, balanced, and free of leakage. The exploratory analysis confirms that tumors introduce measurable changes in both intensity and texture. The recommended preprocessing pipeline should include: corrupt image removal, grayscale conversion (or channel averaging), CLAHE, resizing to a fixed resolution, and normalization. Data augmentation (rotation, flipping) is appropriate given the visual symmetry of brain MRIs.